# KRISIS v0 — Foundation

**Your Smart Triage System.** This notebook works through *every* aspect of the **v0 / Day 1** scope from the project plan, end to end:

1. Domain story finalized
2. Ticket format & output JSON schema defined
3. Category + team taxonomy defined
4. Priority rules defined
5. Few-shot prompt drafted with the three required edge cases (angry tone, vague/short, ambiguous)
6. One raw LLM API call made "by hand" to confirm the mechanism works end to end

> **Scope note:** v0 is about *locking the design* and *proving the mechanism once*. No FastAPI, no Streamlit, no database yet — those arrive in v1. The only cell that needs network access or an API key (and costs money) is the "raw API call" in section 6.

## 0. Setup

Recommended: run this notebook inside a virtual environment so you don't install into system Python.

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install openai pydantic python-dotenv ipykernel
python -m ipykernel install --user --name krisis-v0
```

Then select the **krisis-v0** kernel. Put your key in a `.env` file next to this notebook:

```
OPENAI_API_KEY=sk-...
```

In [ ]:
# Run once if the packages aren't already in this kernel.
%pip install --quiet openai pydantic python-dotenv

In [ ]:
import os
import json
from typing import Literal

from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from a .env file if present
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

## 1. Domain story

KRISIS is a smart triage system for the internal IT and engineering support desk of an AI-first software engineering company. Employees submit free-text tickets covering access requests, infrastructure issues, CI/CD failures, security concerns, developer-tooling problems, and hardware issues.

Today a human reads every ticket and manually decides its **category**, **priority**, and **owning team** before any work starts. That first sorting step is repetitive, slow, and inconsistent between people. KRISIS automates it: given a raw ticket message, it returns a structured decision so the ticket is already sorted by the time a human looks at it.

## 2. Ticket format & output JSON schema

**Input:** a single raw text message, exactly as an employee would type it. Nothing else is required.

**Output:** a JSON object with four fields — `category`, `priority`, `assigned_team`, `reasoning`.

We define this contract as a Pydantic model. The same model doubles as the LLM's structured-output schema in section 6 (and, later, as the FastAPI request/response model).

In [ ]:
Category = Literal[
    "access_iam", "infra_outage", "ci_cd", "security",
    "dev_tooling", "hardware", "unclassified",
]
Priority = Literal["High", "Medium", "Low"]


class TicketDecision(BaseModel):
    """The structured decision KRISIS returns for a single ticket."""
    category: Category = Field(..., description="One of the fixed taxonomy categories")
    priority: Priority = Field(..., description="High / Medium / Low, per the impact rules")
    assigned_team: str = Field(..., description="Owning team for the chosen category")
    reasoning: str = Field(..., description="One line stating which rule drove the decision")


print(json.dumps(TicketDecision.model_json_schema(), indent=2))

## 3. Domain taxonomy

Seven categories, each with a fixed owning team. Kept as data so the prompt, validation, and (later) the API all read from one source of truth.

In [ ]:
TAXONOMY = {
    "access_iam":   {"team": "Identity and access",       "description": "Login, permissions, SSO, credential issues"},
    "infra_outage": {"team": "Infrastructure operations", "description": "Server, network, or cloud infrastructure failures"},
    "ci_cd":        {"team": "Platform engineering",      "description": "Build and deployment pipeline failures"},
    "security":     {"team": "Security team",             "description": "Vulnerabilities, suspicious activity, security incidents"},
    "dev_tooling":  {"team": "Developer experience",      "description": "IDE, internal tools, environment setup issues"},
    "hardware":     {"team": "Hardware support",          "description": "Laptop, peripheral, or device issues"},
    "unclassified": {"team": "Triage",                    "description": "Ticket lacks enough detail to classify confidently"},
}

for cat, info in TAXONOMY.items():
    print(f"{cat:14s} -> {info['team']}")

## 4. Priority rules

Priority is based on **impact and whether the person is blocked**, not on tone or wording. This is the rule that keeps an angry ticket from being scored more urgent than a calm one describing the same problem.

In [ ]:
PRIORITY_RULES = {
    "High":   "Service outage or core function unusable, no workaround, a security incident, or something blocking multiple people or a critical workflow",
    "Medium": "A feature is broken but a workaround exists, the issue affects a single person, or there is a real deadline attached",
    "Low":    "Cosmetic issue, feature request, general question, or anything with no functional impact",
}
for p, rule in PRIORITY_RULES.items():
    print(f"{p:7s}: {rule}")

## 5. Few-shot prompt (with the three required edge cases)

The prompt is few-shot: a system message that states the role, taxonomy, and rules, followed by worked examples. The three required edge cases are covered, and every example's `reasoning` names the rule that drove the decision, so the logic is checkable:

1. **Angry tone** — must not inflate priority.
2. **Very short / vague** — falls back to `unclassified` + Triage.
3. **Ambiguous** — fits more than one category; the reasoning states the tie-break.

In [ ]:
def build_system_prompt() -> str:
    taxonomy_lines = "\n".join(
        f"- {cat}: {info['description']} (team: {info['team']})"
        for cat, info in TAXONOMY.items()
    )
    priority_lines = "\n".join(f"- {p}: {rule}" for p, rule in PRIORITY_RULES.items())
    return f"""You are KRISIS, a smart triage system for an internal IT and engineering support desk.
Given a single raw ticket message, return a structured decision.

Categories (pick exactly one) with their owning team:
{taxonomy_lines}

Priority is based on impact and whether someone is blocked - NOT on tone or wording.
An angry ticket is not more urgent than a calm one describing the same problem.
{priority_lines}

Rules:
- assigned_team MUST be the owning team of the chosen category.
- If the message lacks enough detail to classify confidently, use category "unclassified" and team "Triage".
- reasoning MUST be one line that names the rule driving the decision, not just the outcome.
"""


print(build_system_prompt())

In [ ]:
FEWSHOT = [
    # Edge case 1 - angry tone must NOT inflate priority
    {
        "ticket": "This is RIDICULOUS. I've been locked out of my account for two hours and nobody is helping. Reset my password NOW.",
        "decision": {
            "category": "access_iam",
            "priority": "Medium",
            "assigned_team": "Identity and access",
            "reasoning": "Medium because only one person is blocked and the angry tone does not change impact; category access_iam (account lockout).",
        },
    },
    # Edge case 2 - very short / vague message
    {
        "ticket": "it's broken",
        "decision": {
            "category": "unclassified",
            "priority": "Medium",
            "assigned_team": "Triage",
            "reasoning": "Unclassified and routed to Triage per the insufficient-detail rule: no system, impact, or workaround can be identified.",
        },
    },
    # Edge case 3 - could fit more than one category
    {
        "ticket": "I can't push my code - the CI pipeline rejects my SSH key as unauthorized.",
        "decision": {
            "category": "access_iam",
            "priority": "Medium",
            "assigned_team": "Identity and access",
            "reasoning": "Fits both ci_cd and access_iam; chosen access_iam because the root cause is an unauthorized credential, not a pipeline defect. Medium: one person blocked.",
        },
    },
]
print(len(FEWSHOT), "few-shot examples")

In [ ]:
def build_messages(ticket: str):
    messages = [{"role": "system", "content": build_system_prompt()}]
    for ex in FEWSHOT:
        messages.append({"role": "user", "content": ex["ticket"]})
        messages.append({"role": "assistant", "content": json.dumps(ex["decision"])})
    messages.append({"role": "user", "content": ticket})
    return messages


# Preview the assembled conversation for a sample ticket
for m in build_messages("VPN keeps dropping every few minutes on my laptop."):
    print(m["role"].upper(), ":", m["content"][:110].replace("\n", " "))

## 6. One raw API call, by hand

This is the single cell that needs an `OPENAI_API_KEY` and network access. We send the few-shot prompt plus a real ticket and let OpenAI **structured outputs** return an object that matches `TicketDecision` exactly — no manual JSON parsing, no malformed-output risk at this stage.

> If your `openai` SDK version doesn't have `client.beta.chat.completions.parse`, use `client.chat.completions.parse` (same arguments).

In [ ]:
from openai import OpenAI

MODEL = "gpt-4o-mini"  # cheap default; swap for "gpt-4o" for stronger reasoning
client = OpenAI()      # reads OPENAI_API_KEY from the environment


def classify(ticket: str) -> TicketDecision:
    completion = client.beta.chat.completions.parse(
        model=MODEL,
        messages=build_messages(ticket),
        response_format=TicketDecision,  # <- structured output enforcement
    )
    return completion.choices[0].message.parsed


decision = classify("The whole prod database is down and no one can log in to the app.")
print(decision.model_dump_json(indent=2))

In [ ]:
# Fire the three required edge cases (plus one clear High) through the live model.
samples = [
    "This is RIDICULOUS, reset my password NOW, locked out for hours!",   # angry tone
    "it's broken",                                                         # vague
    "CI pipeline says my SSH key is unauthorized so I can't push",         # ambiguous
    "Prod API is returning 500s for everyone and there is no workaround.", # clear High
]
for s in samples:
    d = classify(s)
    print(f"\nTICKET: {s}\n  -> {d.category} | {d.priority} | {d.assigned_team}\n     {d.reasoning}")

## 7. What v0 confirms — and what's next

If section 6 runs and returns well-formed `TicketDecision` objects with sensible priorities (angry tone stays Medium, "it's broken" becomes unclassified, the SSH-key ticket disambiguates with a stated rule), then **v0 is done**: the domain, schema, taxonomy, priority rules, and prompt are locked, and the mechanism is proven end to end.

**Next (v1):** wrap this logic behind a **FastAPI** backend (`POST /classify`), add retry + fallback reliability, persist every request to **PostgreSQL**, and build the **Streamlit** UI as a client of the API.